In [2]:
# libraries
import mysql.connector
import json
import chromadb
import sqlparse
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [3]:
# CONSTANTS
LLM_API_KEY = ''
DATABASE_NAME = "ecommerce_db"
COLLECTION_NAME = "Table_Schema_of_Ecommerce_Database"
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
LLM_MODEL = 'llama-3.3-70b-versatile'

In [4]:
# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database=f"{DATABASE_NAME}"  # optional
)
cursor = conn.cursor()

# Get the tables in the database
cursor.execute(f"USE {DATABASE_NAME}")
cursor.execute("SHOW TABLES")
tables = [row[0] for row in cursor.fetchall()]

# Get complete schema
def extract_table_info(tables: list):
    metadata = {}

    for table in tables:
        
        # get dolumn details
        cursor.execute(f"DESCRIBE {table}")
        columns_info = []

        for col in cursor.fetchall():
            column_info = {
                "Column": col[0],
                "Type": col[1],
                "Null Constraint": col[2],
                "Key Type": col[3],
                "Default Value": col[4],
                "Extra Constraints": col[5]
            }
            columns_info.append(column_info)

        # get relationship for this table
        query = """
            SELECT 
                TABLE_NAME, COLUMN_NAME,
                CONSTRAINT_NAME,
                REFERENCED_TABLE_NAME,
                REFERENCED_COLUMN_NAME
            FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
            WHERE 
                REFERENCED_TABLE_NAME IS NOT NULL
                AND TABLE_NAME = %s
        """
        cursor.execute(query, (table,))
        relationships = cursor.fetchall()

        metadata[table] = {
            "columns": columns_info,
            "relationships": relationships
        }

    table_info = []
        
    for table, info in metadata.items():
        table_text = f"\nDatabase: ecommerce_db\n"
        table_text += f"Table: {table}\n"

        table_text += f"\nColumns:\n"
        for column in info["columns"]:
            table_text += (
                    f"Column: {column['Column']}, "
                    f"Type: {column['Type']}, "
                    f"Null Constraint: {column['Null Constraint']}, "
                    f"Key Type: {column['Key Type']}, "
                    f"Default Value: {column['Default Value']}, "
                    f"Extra Constraints: {column['Extra Constraints']}\n"
            )
        if info['relationships']:
            table_text += f"\nRelationships:\n"
            for row in info["relationships"]:
                table_text += (
                        f"{row[0]}.{row[1]} → {row[3]}.{row[4]} with (constraint: {row[2]})\n"
                )
        else:
            table_text += f"\nRelationships: None\n"

        table_info.append(table_text)    

    return table_info

info = extract_table_info(tables)

In [5]:
# Vector Database creation
client = chromadb.PersistentClient()
# collection creation
try:
    client.delete_collection(name = collection_name)
except:
    pass
    
collection = client.get_or_create_collection(
    name = COLLECTION_NAME,
    embedding_function= SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
)

# add records to the collection
ids = [f"id{i+1}" for i in range(len(info))]
collection.add(
    ids=ids,
    documents=info
)

# user query
user_query = "Qeury to list all the persons with max sales year over year"

# Query Vector Database
query = collection.query(
    query_texts=user_query,
    n_results=4
)

# Combine all the chunks
document_text = '\n\n'.join(query['documents'][0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# LLM model creation
# llama model
llm  = ChatGroq(
    temperature=0,
    api_key= LLM_API_KEY,
    model=LLM_MODEL,
    max_tokens= 3000
)

# prompt
prompt = PromptTemplate.from_template(
    """
    You are an expert MySQL SQL Assistant.

    Your task is to generate optimized, production-ready SQL queries.
    
    Rules:
    - Only generate SELECT queries
    - Never generate DELETE, UPDATE, INSERT, DROP, ALTER, or TRUNCATE queries
    - Use explicit column names instead of SELECT *
    - Use only the tables and columns provided in the schema
    - Do not hallucinate tables or columns
    - Generate syntactically correct MySQL queries
    - Prefer optimized joins and filtering
    - If the query cannot be generated from the schema, explain why
    
    Return the output STRICTLY in this format ONLL: 
    <SQL QUERY HERE>",
    
    DO NOT:
    - Return markdown
    - Return backticks
    - Return extra text
    - Return comments outside JSON
    
    User Question:
    {user_question}

    Content:
    {extracted_content}
    """
)

# chain creation
chain = prompt | llm

# response generation and display
response = chain.invoke(
    {
        "extracted_content": document_text,
        "user_question": user_query
    }
)

# formatting the sql 
formatted_sql = sqlparse.format(
    response.content,
    reindent=True,
    keyword_case="upper"
)

print(formatted_sql)

SELECT o.customer_id,
       c.name,
       SUM(oi.line_total) AS total_sales
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id,
         c.name
ORDER BY total_sales DESC
LIMIT 1
